# Semantic Retrieval Demo

This notebook demonstrates how to:
1. Extract sentences from MD files with source tracking
2. Generate embeddings using Qwen3-Embedding:8B via Ollama
3. Store embeddings in FAISS for fast similarity search
4. Perform semantic search with cosine similarity
5. Synthesize answers using an LLM

**Key Feature**: Incremental processing - only new files are embedded and saved.

## Setup and Dependencies

In [1]:
# Install required packages
%pip install ollama faiss-cpu nltk numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ollama
import numpy as np
import faiss
import nltk
import json
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, asdict

# Download NLTK data for sentence tokenization
nltk.download('punkt_tab', quiet=True)

True

## Setup

Make sure you have Ollama installed with these models:
```bash
ollama pull qwen3-embedding:8b
ollama pull llama3:8b
```

In [3]:
@dataclass
class SentenceWithSource:
    """Container for a sentence with its source information"""
    text: str
    file_path: str
    file_title: str  # MD file title for tracking
    line_number: int
    section_header: str = ""

## Helper Functions

In [4]:
# This uses qwen3-embedding:8b model to generate a 4096-dimension embedding for a sentence
def get_embedding(text: str, model: str = "qwen3-embedding:8b") -> np.ndarray:
    """Get embedding vector for text using Ollama
    
    Args:
        text (str): The text to generate an embedding for
        model (str, optional): The model to use. Defaults to "qwen3-embedding:8b".
    
    Returns:
        np.ndarray: The embedding vector
    """
    try:
        response = ollama.embeddings(model=model, prompt=text)
        return np.array(response['embedding'], dtype=np.float32)
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return None

In [5]:
def split_into_sentences(text: str, file_path: str, file_title: str) -> List[SentenceWithSource]:
    """Split text into sentences while tracking source information
    
    Args:
        text (str): The text to split into sentences
        file_path (str): The path to the file the text is from
        file_title (str): The title of the file
    
    Returns:
        List[SentenceWithSource]: A list of sentences with source information
    """
    sentences_with_source = []
    lines = text.split('\n')
    current_section = ""
    
    for line_num, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue
            
        # Track section headers
        if line.startswith('#'):
            current_section = line.strip('#').strip()
            continue
        
        # Skip tables, images, and short lines
        if '|' in line or line.startswith('---') or line.startswith('![]') or line.startswith('Fig.'):
            continue
        
        # Split into sentences
        sentences = nltk.sent_tokenize(line)
        
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 20:  # Filter short sentences
                sentences_with_source.append(
                    SentenceWithSource(
                        text=sentence,
                        file_path=file_path,
                        file_title=file_title,
                        line_number=line_num,
                        section_header=current_section
                    )
                )
    
    return sentences_with_source

## Storage Configuration

In [6]:
# Storage paths
STORAGE_DIR = Path("vector_store")
STORAGE_DIR.mkdir(exist_ok=True)

EMBEDDINGS_FILE = STORAGE_DIR / "embeddings.npy"
FAISS_INDEX_FILE = STORAGE_DIR / "index.faiss"
METADATA_FILE = STORAGE_DIR / "metadata.json"

print(f"Storage directory: {STORAGE_DIR.absolute()}")

Storage directory: c:\Users\JR\Documents\GitHub\crag-doc-reader\vector_store


In [7]:
def load_existing_data():
    """Load existing embeddings, index, and metadata if they exist
    
    Args:
        None
    
    Returns:
        Tuple[List[SentenceWithSource], np.ndarray, faiss.Index, Set[str]]: A tuple containing the sentences, embeddings, index, and processed files
    """
    if METADATA_FILE.exists():
        with open(METADATA_FILE, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        embeddings = np.load(EMBEDDINGS_FILE) if EMBEDDINGS_FILE.exists() else None
        index = faiss.read_index(str(FAISS_INDEX_FILE)) if FAISS_INDEX_FILE.exists() else None
        
        # Reconstruct sentences from metadata
        sentences = [
            SentenceWithSource(**item) for item in metadata['sentences']
        ]
        processed_files = set(metadata.get('processed_files', []))
        
        print(f"Loaded {len(sentences)} sentences from {len(processed_files)} files")
        return sentences, embeddings, index, processed_files
    
    return [], None, None, set()

In [8]:
def save_data(sentences: List[SentenceWithSource], embeddings: np.ndarray, 
              index: faiss.Index, processed_files: set):
    """Save embeddings, FAISS index, and metadata
    
    Args:
        sentences (List[SentenceWithSource]): The sentences to save
        embeddings (np.ndarray): The embeddings to save
        index (faiss.Index): The FAISS index to save
        processed_files (set): The processed files to save
    """
    # Save embeddings
    np.save(EMBEDDINGS_FILE, embeddings)
    
    # Save FAISS index
    faiss.write_index(index, str(FAISS_INDEX_FILE))
    
    # Save metadata
    metadata = {
        'sentences': [asdict(s) for s in sentences],
        'processed_files': list(processed_files),
        'embedding_model': 'qwen3-embedding:8b',
        'dimension': embeddings.shape[1],
        'total_sentences': len(sentences)
    }
    
    with open(METADATA_FILE, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Saved {len(sentences)} sentences, embeddings, and FAISS index")

## Process MD Files (Incremental)

In [9]:
# Load existing data
sentences, embeddings, index, processed_files = load_existing_data()

print(f"Previously processed files: {processed_files}")

Loaded 350 sentences from 1 files
Previously processed files: {'1924 Mayor Aua transect Structure'}


In [10]:
# Specify MD file to process
md_file_path = r"output\1924 Mayor Aua transect Structure\1924 Mayor Aua transect Structure.md"
file_path = Path(md_file_path).resolve()
file_title = file_path.stem  # Use filename without extension as title

print(f"File: {file_path}")
print(f"Title: {file_title}")

# Check if already processed
if file_title in processed_files:
    print(f"⚠️ '{file_title}' already processed. Skipping...")
else:
    print(f"✓ New file - will process")

File: C:\Users\JR\Documents\GitHub\crag-doc-reader\output\1924 Mayor Aua transect Structure\1924 Mayor Aua transect Structure.md
Title: 1924 Mayor Aua transect Structure
⚠️ '1924 Mayor Aua transect Structure' already processed. Skipping...


In [11]:
# Process new file if not already done
if file_title not in processed_files:
    # Read file
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract sentences
    new_sentences = split_into_sentences(content, str(file_path), file_title)
    print(f"Extracted {len(new_sentences)} sentences")
    
    # Generate embeddings
    print("Generating embeddings...")
    new_embeddings = []
    for i, sentence in enumerate(new_sentences):
        if i % 10 == 0:
            print(f"  {i+1}/{len(new_sentences)}")
        
        embedding = get_embedding(sentence.text)
        if embedding is not None:
            new_embeddings.append(embedding)
        else:
            # Fallback to zero vector
            print(f"  {i+1}/{len(new_sentences)}: Failed to generate embedding")
            new_embeddings.append(np.zeros(4096, dtype=np.float32))
    
    new_embeddings = np.vstack(new_embeddings)
    
    # Normalize for cosine similarity
    faiss.normalize_L2(new_embeddings)
    
    # Merge with existing data
    if embeddings is not None:
        embeddings = np.vstack([embeddings, new_embeddings])
        sentences.extend(new_sentences)
    else:
        embeddings = new_embeddings
        sentences = new_sentences
    
    # Rebuild FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    # Mark as processed
    processed_files.add(file_title)
    
    # Save everything
    save_data(sentences, embeddings, index, processed_files)
    
    print(f"✓ Processed and saved. Total: {len(sentences)} sentences from {len(processed_files)} files")

## Semantic Search

In [12]:
def search(query: str, top_k: int = 5) -> List[Tuple[SentenceWithSource, float]]:
    """Search for most similar sentences to query
    
    Args:
        query (str): The query to search for
        top_k (int, optional): The number of results to return. Defaults to 5.
    
    Returns:
        List[Tuple[SentenceWithSource, float]]: A list of tuples containing the most similar sentences and their scores
    """
    if index is None:
        raise ValueError("No index loaded. Process files first.")
    
    # Get and normalize query embedding
    query_embedding = get_embedding(query)
    if query_embedding is None:
        return []
    
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(sentences):
            results.append((sentences[idx], float(score)))
    
    return results

In [13]:
# Test search
query = "What is the structure of Samoan reefs?"
results = search(query, top_k=5)

print(f"Query: {query}\n")
print("Top 5 Results:")
print("=" * 60)

for i, (sentence, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f} | File: {sentence.file_title}")
    print(f"   Section: {sentence.section_header}")
    print(f"   Text: {sentence.text[:120]}...")
    print()

Query: What is the structure of Samoan reefs?

Top 5 Results:
1. Score: 0.7328 | File: 1924 Mayor Aua transect Structure
   Section: II. ECOLOGY OF THE AUA CORAL-REEF, PAGO PAGO HARBOR.
   Text: While in Samoa we observed some of the vicissitudes to which coral-reefs may be subjected....

2. Score: 0.6780 | File: 1924 Mayor Aua transect Structure
   Section: STRUCTURE OF THE SAMOAN REEFS.
   Text: TABLE 2.—Soundings off seaward edges of fringing reefs of Tutuila Island, Samoa....

3. Score: 0.6747 | File: 1924 Mayor Aua transect Structure
   Section: STRUCTURE OF THE SAMOAN REEFS.
   Text: With the exception of Rose Island, which is an atoll-rim composed chiefly of lithothamnium, all the Samoan Islands are v...

4. Score: 0.6733 | File: 1924 Mayor Aua transect Structure
   Section: LITERATURE CITED.
   Text: The structure and distribution of coral reefs....

5. Score: 0.6728 | File: 1924 Mayor Aua transect Structure
   Section: II. ECOLOGY OF THE AUA CORAL-REEF, PAGO PAGO HARBOR.
   Te

In [14]:
def synthesize_answer(query: str, results: List[Tuple[SentenceWithSource, float]], 
                      model: str = "llama3:8b") -> str:
    """Use LLM to synthesize answer from relevant sentences
    
    Args:
        query (str): The query to answer
        results (List[Tuple[SentenceWithSource, float]]): The results to use
        model (str, optional): The model to use. Defaults to "llama3:8b".
    
    Returns:
        str: The answer
    """
    context = "\n\n".join([
        f"[{sent.file_title}] {sent.section_header}\n{sent.text}"
        for sent, score in results
    ])
    
    prompt = f"""Based on the following context, answer the question. Be specific and cite sources.

Context:
{context}

Question: {query}

Answer:"""
    
    try:
        response = ollama.generate(model=model, prompt=prompt)
        return response['response']
    except Exception as e:
        return f"Error: {e}"

In [15]:
# Generate answer using LLM
query = "What is the structure of Samoan reefs?"
results = search(query, top_k=5)

print(f"Query: {query}\n")
answer = synthesize_answer(query, results)

print("Answer:")
print("=" * 60)
print(answer)
print("=" * 60)

Query: What is the structure of Samoan reefs?

Answer:
Based on the context, the structure of Samoan reefs is described as follows:

* The majority of Samoan Islands are volcanic.
* With the exception of Rose Island, which is an atoll-rim composed chiefly of lithothamnium (a type of coral).
* The soundings off seaward edges of fringing reefs of Tutuila Island, Samoa, were taken and presented in a table (Table 2).

Source:
[1924 Mayor Aua transect Structure] II. ECOLOGY OF THE AUA CORAL-REEF, PAGO PAGO HARBOR.

Note: The reference to Table 2 does not provide further information on the structure of Samoan reefs beyond the soundings taken off the seaward edges of fringing reefs.


In [16]:
# Test search
query = "What is the structure of Samoan reefs?"
results = search(query, top_k=5)

print(f"Query: {query}\n")
print("Top 5 Results:")
print("=" * 60)

for i, (sentence, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   File: {sentence.file_title}")
    print(f"   Section: {sentence.section_header}")
    print(f"   Text: {sentence.text}...")
    print()

Query: What is the structure of Samoan reefs?

Top 5 Results:
1. Score: 0.7328
   File: 1924 Mayor Aua transect Structure
   Section: II. ECOLOGY OF THE AUA CORAL-REEF, PAGO PAGO HARBOR.
   Text: While in Samoa we observed some of the vicissitudes to which coral-reefs may be subjected....

2. Score: 0.6780
   File: 1924 Mayor Aua transect Structure
   Section: STRUCTURE OF THE SAMOAN REEFS.
   Text: TABLE 2.—Soundings off seaward edges of fringing reefs of Tutuila Island, Samoa....

3. Score: 0.6747
   File: 1924 Mayor Aua transect Structure
   Section: STRUCTURE OF THE SAMOAN REEFS.
   Text: With the exception of Rose Island, which is an atoll-rim composed chiefly of lithothamnium, all the Samoan Islands are volcanic....

4. Score: 0.6733
   File: 1924 Mayor Aua transect Structure
   Section: LITERATURE CITED.
   Text: The structure and distribution of coral reefs....

5. Score: 0.6728
   File: 1924 Mayor Aua transect Structure
   Section: II. ECOLOGY OF THE AUA CORAL-REEF, PAGO PAGO 